# Road Safety - Data Cleaning and Clustering

This notebook cleans the WHO Global Status Report on Road Safety 2023 dataset and runs K-Means clustering on it. The cleaning logic is wrapped in a reusable function near the end so the Streamlit app can use the same pipeline when a user uploads their own country-level dataset.

The cleaned outputs are saved to `data/processed/` for the other two notebooks (PCA + Anomaly Detection, Time Series Forecasting) and the final app to use.

### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy import stats

sns.set_style('whitegrid')

### Load the file

The Excel file has two header rows. The first row contains short codes like CP_0, CP_1, and the second row has the actual column names we want to use. We tell pandas to use row index 1 as the header, then drop the leftover code row that ends up at the top of the data.

In [ ]:
RAW_PATH = 'data/raw/gsrrs23-indicators-for-participating-countries-or-territories.xlsx'

df = pd.read_excel(RAW_PATH, sheet_name='Indicators', header=1)
df = df.iloc[1:].reset_index(drop=True)

df.shape

In [ ]:
df.head(3)

### What am I doing here and why

Before touching the data I want to understand what is actually inside it. The summary table below lists every column, its data type, how many values are filled in, how many are missing, and what percent that missing share is. This tells me which columns are clean enough to use directly and which ones I will have to drop or fill in later.

I am also pulling out just the label columns separately so I can confirm that every row really is one country, with proper identification (ISO code, name, region, income group).

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'missing': df.isna().sum(),
    'missing_%': (df.isna().mean() * 100).round(1)
})
summary.head(20)

The data has 171 countries and 228 columns. The first columns are basic info: country name, ISO code, population, income group, and WHO region. These are almost all filled in - only Income group has 3 missing values.

Next come the GRSSR participation columns. These show if a country joined the WHO road safety report in 2009, 2013, 2015, and 2018. Most countries joined, so only 5-7 percent are missing.

After that come the real road safety indicators. Total reported fatalities is almost complete, but when we look at the details - deaths by gender, deaths by road user type like pedestrian or cyclist - the missing values jump to 12-22 percent. This makes sense because not every country tracks deaths in that much detail.

The pattern is clear: simple info is filled in, but the more detailed a column gets, the more countries leave it empty. We will handle this in the missing values step below.

In [ ]:
df[['ISO_3 country name', 'Country name', 'Population', 'Income group', 'WHO Region']].head()

### Are there any duplicate countries?

If the same country appeared twice in the data it would pull the clustering toward that point and give a wrong result. To be safe, we check three different things: are any full rows identical, do any ISO codes repeat, and do any country names repeat. All three checks need to return zero.

In [ ]:
print('duplicate rows:        ', df.duplicated().sum())
print('duplicate ISO codes:  ', df['ISO_3 country name'].duplicated().sum())
print('duplicate names:      ', df['Country name'].duplicated().sum())

All three returned zero. No country appears twice. We can move on.

### Missing values - the big picture

A lot of WHO indicators are missing for a lot of countries. The first thing I want to know is how widespread the problem is across all 228 columns - not just the first 20 I looked at above. The numbers below count how many columns fall into each missing range, and the histogram visualises the same thing.

In [ ]:
miss_pct = df.isna().mean().sort_values(ascending=False) * 100

print('columns with more than 50% missing:', (miss_pct > 50).sum())
print('columns with more than 70% missing:', (miss_pct > 70).sum())
print('columns with 0 missing:            ', (miss_pct == 0).sum())
print('total columns:                     ', len(df.columns))

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(miss_pct, bins=30, edgecolor='black')
plt.axvline(50, color='red', linestyle='--', label='50% cutoff')
plt.xlabel('% missing')
plt.ylabel('number of columns')
plt.title('How empty each column is')
plt.legend()
plt.show()

The histogram shows a clear pattern. Most columns are either almost full (the tall bar on the left) or quite empty (the bars on the right). About 60 columns are more than half empty. Those columns cannot be trusted - filling in more than half of the values with our own guesses would make the cluster results meaningless. We will drop them in the next step.

The red dashed line at 50 percent is the cutoff we are using. Everything to the right of it gets dropped.

### Fix Palestine's name

The WHO file labels Palestine as "occupied Palestinian territory, including east Jerusalem". The IHME time series file in our other notebook just calls it "Palestine". If those names do not match, we will not be able to join the two datasets together when we build the Streamlit app.

The fix is to map every variant of the name to one clean version. I am also including other variants that show up in different public datasets in case the user uploads one of those later through the app.

In [ ]:
COUNTRY_NAME_MAP = {
    'occupied Palestinian territory, including east Jerusalem': 'Palestine',
    'occupied Palestinian territory': 'Palestine',
    'State of Palestine': 'Palestine',
    'Palestinian Territory': 'Palestine',
    'West Bank and Gaza Strip': 'Palestine',
}

df['Country name'] = df['Country name'].replace(COUNTRY_NAME_MAP)
df[df['ISO_3 country name'] == 'PSE'][['ISO_3 country name', 'Country name']]

Palestine is now labeled cleanly. The same map will be used by the other notebooks and by the upload feature.

### Drop the mostly-empty columns

Now we apply the 50 percent cutoff decision from the missing values step above. Any column with more than half its values missing gets dropped from the dataframe.

In [ ]:
cols_to_drop = miss_pct[miss_pct > 50].index.tolist()
df_clean = df.drop(columns=cols_to_drop)

print('columns before:', df.shape[1])
print('columns after: ', df_clean.shape[1])
print('dropped:       ', len(cols_to_drop))

### Separate the labels from the features

K-Means is a numerical algorithm - it cannot work with text. So we need to split the columns into two groups:

The label columns (ISO code, country name, income group, WHO region) describe each country but will not be used in clustering. They get kept aside so we can attach them back to the results later for interpretation.

The feature columns are the numeric ones - all the road safety indicators that K-Means will use to measure how similar two countries are.

In [ ]:
LABEL_COLS = ['ISO_3 country name', 'Country name', 'Income group', 'WHO Region']

numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()
print('numeric feature columns:', len(numeric_cols))

### Fill the remaining missing values

After dropping the very-empty columns, the columns we kept still have some missing values scattered around. We need to fill those before we can do any math on the data.

The choice here is between mean and median imputation. Mean is the average and gets pulled toward extreme values - if one country has a huge number, the mean shifts. Median is the middle value and ignores extremes. Since our data has very different countries (tiny island nations next to large populous ones), the median is the safer choice. It gives us a typical value rather than an inflated one.

In [ ]:
features = df_clean[numeric_cols].copy()

print('missing before imputation:', features.isna().sum().sum())
features = features.fillna(features.median(numeric_only=True))
print('missing after imputation: ', features.isna().sum().sum())

Zero missing values left. Every country now has a complete row of numbers.

### Check for outliers before scaling

K-Means is sensitive to extreme values because it uses distance to assign countries to clusters. Before we scale and cluster, we need to know if any features have wildly extreme values that would dominate the distance calculation.

We check this two ways. First a boxplot of the features with the highest variance - if any feature has dots floating far above the box, that feature has outliers. Second a skewness score for every feature - skewness near 0 means symmetric and well behaved, skewness above 2 or 3 means heavy tail, and skewness above 5 means severely skewed.

In [ ]:
# pick the 10 features with the highest variance for the boxplot
top_var = features.var().sort_values(ascending=False).head(10).index

plt.figure(figsize=(14, 6))
features[top_var].boxplot(rot=45)
plt.title('Boxplot of the 10 highest-variance features')
plt.ylabel('value')
plt.yscale('log')  # log scale so we can see all features on one plot
plt.tight_layout()
plt.show()

The log scale is needed because the features live on completely different scales - that fact alone is already a warning. On the plot, any feature where the dots sit far above the box has outliers. Look especially at Population, which is famous for having a few enormous countries (China, India) and many small ones.

In [ ]:
# skewness table - sorted from worst to best
skew_scores = features.apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)

skew_table = pd.DataFrame({
    'skewness': skew_scores,
    'severity': pd.cut(skew_scores,
                       bins=[-np.inf, -5, -2, 2, 5, np.inf],
                       labels=['severe negative', 'high negative', 'normal', 'high positive', 'severe positive'])
})

skew_table.head(15)

The skewness table shows the 15 most skewed features. Anything in the "severe positive" category is a strong outlier driver and will hurt the clustering. We need to look at this output and decide what to do:

- If only one or two features are severely skewed and they are not central to road safety analysis (like Population, which is a size variable not a safety variable), we can drop them.
- If many features are skewed, we can apply a log transformation to compress the range.
- If a few features are skewed but central to the analysis, we keep them and accept that the cluster results will reflect them.

Run the cell above and look at the table. The decision for what to do goes in the next cell.

### Decision: handle the outliers

After looking at the boxplot and the skewness table, the action we take goes here. Fill this in based on what the data actually shows you.

Common case for this dataset: Population is by far the most skewed feature, and it is a size variable rather than a safety variable. Two big countries (China, India) dominate it and pull K-Means toward an "outlier vs everyone else" split. The fix is to drop Population from the feature list - we still keep it as a label column for context, but it does not drive the clustering.

In [ ]:
# drop Population from the features (keeps it available as a label)
DROP_FROM_FEATURES = ['Population']

features_no_outlier = features.drop(columns=[c for c in DROP_FROM_FEATURES if c in features.columns])

print('features before drop:', features.shape[1])
print('features after drop: ', features_no_outlier.shape[1])

### Verify outliers improved

Quick check that dropping the outlier driver actually helped. The new boxplot should show features that are more comparable in scale, and the new skewness table should have fewer entries in the severe range.

In [ ]:
top_var_after = features_no_outlier.var().sort_values(ascending=False).head(10).index

plt.figure(figsize=(14, 6))
features_no_outlier[top_var_after].boxplot(rot=45)
plt.title('Boxplot after dropping the outlier driver')
plt.ylabel('value')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
skew_after = features_no_outlier.apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)
print('severe positive (skew > 5):', (skew_after > 5).sum())
print('high positive (2 < skew <= 5):', ((skew_after > 2) & (skew_after <= 5)).sum())
print('normal (-2 <= skew <= 2):', ((skew_after >= -2) & (skew_after <= 2)).sum())

### Scale the features

K-Means measures similarity between countries using distance. If one column is measured in millions and another is a percentage (0 to 100), the column with the bigger numbers will completely dominate the distance calculation - even if it is not actually more important.

StandardScaler fixes this by transforming every column to have mean 0 and standard deviation 1. After this step, every feature contributes equally to the distance calculation regardless of its original scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_no_outlier)

print('scaled shape:', X_scaled.shape)
print('first column mean (should be ~0):', round(X_scaled[:, 0].mean(), 4))
print('first column std  (should be ~1):', round(X_scaled[:, 0].std(), 4))

### Find a good k

K-Means needs us to tell it how many clusters to make. We do not know the right number in advance, so we try several and measure each one with two metrics.

Inertia (the elbow method) measures how tight the clusters are - lower is better, but inertia always drops as k grows, so we look for the "elbow" point where adding more clusters stops helping much.

Silhouette score measures how well separated the clusters are. It ranges from -1 to 1, where higher is better. Above 0.5 is considered strong separation, 0.25 to 0.5 is reasonable, and below 0.25 means the clusters overlap a lot.

If the two metrics agree on a value of k, that is our answer. If they disagree, we lean on silhouette because it directly measures cluster quality.

In [ ]:
k_values = range(2, 9)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

pd.DataFrame({'k': list(k_values), 'inertia': inertias, 'silhouette': silhouettes}).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(list(k_values), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k')
axes[0].set_ylabel('inertia')
axes[0].set_title('Elbow method')

axes[1].plot(list(k_values), silhouettes, 'o-', color='darkorange')
axes[1].set_xlabel('k')
axes[1].set_ylabel('silhouette score')
axes[1].set_title('Silhouette method')

plt.tight_layout()
plt.show()

Pick the k where the silhouette score is highest, then check that it also looks like a reasonable elbow point on the left plot. Set that value below.

### Fit K-Means with the chosen k

Now we run K-Means one final time with the value of k we picked above. The result is a cluster label (0, 1, 2 ...) for every country in the dataset.

In [ ]:
K_FINAL = 4  # change this based on the silhouette plot above

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

print('cluster sizes:')
print(pd.Series(cluster_labels).value_counts().sort_index())

### Visualize the clusters in 2D

We cannot draw 90-something dimensions on a page, so we use PCA to project the data down to 2 dimensions just for plotting. Each dot is a country, the color shows which cluster K-Means put it in, and Palestine is marked with a red star so we can see exactly where it lands.

Note: this 2D PCA is for plotting only. The full PCA analysis happens in Osama's notebook.

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
coords = pca_2d.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Cluster': cluster_labels,
    'Country': df_clean['Country name'].values,
    'ISO': df_clean['ISO_3 country name'].values,
})
plot_df.head()

In [ ]:
plt.figure(figsize=(12, 8))

palette = sns.color_palette('Set2', K_FINAL)
for c in range(K_FINAL):
    sub = plot_df[plot_df['Cluster'] == c]
    plt.scatter(sub['PC1'], sub['PC2'],
                color=palette[c], label=f'Cluster {c}',
                s=70, alpha=0.7, edgecolor='black', linewidth=0.5)

pal = plot_df[plot_df['ISO'] == 'PSE']
plt.scatter(pal['PC1'], pal['PC2'],
            marker='*', s=400, color='red',
            edgecolor='black', linewidth=1.5, label='Palestine', zorder=5)

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% of variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% of variance)')
plt.title(f'Country clusters in 2D (k = {K_FINAL})')
plt.legend()
plt.tight_layout()
plt.show()

### What is in each cluster

A cluster number on its own (0, 1, 2, 3) is meaningless. We need to look inside each cluster and see what kinds of countries it contains. We do this two ways - by income group and by WHO region. If most rich countries land in one cluster and most poor countries land in another, that tells us K-Means is picking up the economic-development axis. If clusters split along regions, we are seeing geographic patterns.

In [ ]:
profile_df = df_clean[LABEL_COLS].copy()
profile_df['Cluster'] = cluster_labels

print('Cluster vs Income Group:')
pd.crosstab(profile_df['Cluster'], profile_df['Income group'])

In [ ]:
print('Cluster vs WHO Region:')
pd.crosstab(profile_df['Cluster'], profile_df['WHO Region'])

### Top features separating the clusters

Beyond income and region, we want to know which actual road safety features are driving the cluster split. We compute the mean of each feature inside each cluster, then look at which features vary the most across clusters - those are the strongest separators.

In [ ]:
cluster_means = features_no_outlier.copy()
cluster_means['Cluster'] = cluster_labels
group_means = cluster_means.groupby('Cluster').mean()

variation = group_means.std().sort_values(ascending=False)
top_distinguishing = variation.head(10).index

group_means[top_distinguishing].round(2)

### Where Palestine lands

The whole project has a Palestinian spotlight angle. So once we have the cluster labels, we want to immediately know which cluster Palestine is in and which other countries share that cluster - those are Palestine's road safety peers.

In [ ]:
pal_cluster = profile_df.loc[profile_df['ISO_3 country name'] == 'PSE', 'Cluster'].values[0]
print('Palestine is in cluster:', pal_cluster)

neighbors = profile_df[profile_df['Cluster'] == pal_cluster]['Country name'].tolist()
print(f'\nCountries in the same cluster as Palestine ({len(neighbors)} total):')
for n in sorted(neighbors):
    print(' -', n)

### Reusable cleaning function for the upload feature

The Streamlit app lets users upload their own country-level dataset. When that happens, the app needs to run the same cleaning pipeline we just walked through. Rather than duplicate code, we wrap the whole pipeline in one function that takes a raw dataframe and returns the cleaned features matrix, the labels, and the cluster labels.

The function works on any country-level dataset that has at least a country name column. It auto-detects which columns are numeric, applies the same cleaning steps (drop empty columns, normalize country names, fill missing with median, drop outlier-driver columns, scale), and runs K-Means.

In [ ]:
def clean_country_dataset(raw_df, country_col='Country name', missing_threshold=0.5, k=None):
    """
    Run the same cleaning pipeline used on the WHO data on any uploaded
    country-level dataframe.

    Parameters
    ----------
    raw_df : pandas DataFrame
        The user's uploaded data.
    country_col : str
        Name of the column that holds the country name.
    missing_threshold : float
        Drop any column with more than this fraction of missing values.
    k : int or None
        Number of clusters. If None, the function picks the k with the
        highest silhouette score.

    Returns
    -------
    dict with keys: features, labels_df, cluster_labels, k_used,
                    scaler, dropped_cols
    """
    df_work = raw_df.copy()

    # 1. normalize country names if our map applies
    if country_col in df_work.columns:
        df_work[country_col] = df_work[country_col].replace(COUNTRY_NAME_MAP)

    # 2. drop mostly-empty columns
    miss_share = df_work.isna().mean()
    dropped = miss_share[miss_share > missing_threshold].index.tolist()
    df_work = df_work.drop(columns=dropped)

    # 3. pick numeric features, fill missing with median
    num = df_work.select_dtypes(include='number').copy()
    num = num.fillna(num.median(numeric_only=True))

    # 4. drop columns with extreme skewness (severe outlier drivers)
    if len(num) > 0:
        skew_vals = num.apply(lambda x: stats.skew(x.dropna()) if x.notna().any() else 0)
        outlier_cols = skew_vals[skew_vals > 5].index.tolist()
        num = num.drop(columns=outlier_cols)
    else:
        outlier_cols = []

    # 5. scale
    scl = StandardScaler()
    X = scl.fit_transform(num)

    # 6. pick k by silhouette if not given
    if k is None and len(X) >= 4:
        best_k, best_score = 2, -1
        for trial_k in range(2, min(9, len(X))):
            km = KMeans(n_clusters=trial_k, random_state=42, n_init=10)
            lbl = km.fit_predict(X)
            score = silhouette_score(X, lbl)
            if score > best_score:
                best_k, best_score = trial_k, score
        k = best_k

    # 7. fit final
    final_km = KMeans(n_clusters=k, random_state=42, n_init=10)
    final_labels = final_km.fit_predict(X)

    # 8. build labels frame (whatever non-numeric cols are available)
    label_cols_available = [c for c in df_work.columns if c not in num.columns]
    labels_df = df_work[label_cols_available].copy()
    labels_df['Cluster'] = final_labels

    return {
        'features': pd.DataFrame(X, columns=num.columns),
        'labels_df': labels_df,
        'cluster_labels': final_labels,
        'k_used': k,
        'scaler': scl,
        'dropped_cols': dropped,
        'outlier_cols': outlier_cols,
    }

Quick test - run the function on the original raw WHO dataframe and confirm the results match what we got going through the steps manually.

In [ ]:
result = clean_country_dataset(df)

print('k chosen:', result['k_used'])
print('feature columns kept:', result['features'].shape[1])
print('countries clustered: ', len(result['cluster_labels']))
print('empty columns dropped:', len(result['dropped_cols']))
print('outlier columns dropped:', result['outlier_cols'])

### Save the outputs

Two files get saved for the rest of the project to consume.

country_features.csv is the cleaned feature matrix - Osama uses this in his PCA and anomaly detection notebook.

country_clusters.csv is the country-to-cluster mapping - the Streamlit map page uses this to color countries by their cluster.

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

features_out = df_clean[LABEL_COLS].copy()
features_out = pd.concat([features_out, features_no_outlier.reset_index(drop=True)], axis=1)
features_out.to_csv('data/processed/country_features.csv', index=False)

clusters_out = profile_df.copy()
clusters_out.to_csv('data/processed/country_clusters.csv', index=False)

print('saved country_features.csv:', features_out.shape)
print('saved country_clusters.csv:', clusters_out.shape)